# Module 5: Geocoding


## Learning Objectives
At the end of this module, you should be able to:
- geocode addresses, i.e., convert a list of addresses into a list of coordinates

## Geocoding

Geocoding is the process of transforming place names or addresses into coordinates (and vice versa). In this section, you will learn how to geocode addresses using **geopandas** and **geopy** libraries. Geopy and other geocoding libraries (such as [geocoder](http://geocoder.readthedocs.io/)) make it easy to locate the coordinates of addresses, cities, countries, and landmarks across the globe. In practice, geocoding libraries are often based on **Application Programming Interfaces (APIs)** where you can send requests, and receive responses in the form of place names, addresses and coordinates. Geopy offers access to several geocoding services, such as Photon and Nominatim that rely on data from OpenStreetMap, among various other services. You can see a full list of supported geocoding services and usage details from the [geopy documentation](https://geopy.readthedocs.io/en/stable/).

It is important to pay attention to the geocoding providers' Terms of Use. Geocoding services might require an API key in order to use them which means that you need to register for the service before you can access results from their API. Furthermore, rate limiting also restricts the use of these services. The geocoding process might end up in an error if you are making too many requests in a short time period (such as when trying to geocode large number of addresses). If you pay for the geocoding service, you can naturally make more requests to the API.

Next, you will learn how to use the **Nominatim** geocoder for locating a relatively small number of addresses. The Nominatim API is not meant for super heavy use. Nominatim doesn't require the use of an API key, but the usage of the Nominatim service is rate-limited to 1 request per second (3600 / hour). Users also need to provide an identifier for their application, and give appropriate attribution to using OpenStreetMap data. You can read more about Nominatim usage policy from their [documentation](https://operations.osmfoundation.org/policies/nominatim/). When using Nominatim via geopandas and geopy, we can specify a custom **user_agent** parameter to identify our application, and we can add a **timeout** to allow enough time to get the response from the service.  

We will geocode addresses stored in a text file called **Michigan_colleges.txt** for a list of Michigan colleges or universities. The first rows of the data look like this:

```
id;addr
1;500 S State St, Ann Arbor, MI 48109
2;220 Trowbridge Rd, East Lansing, MI 48824
3;42 W Warren Ave, Detroit, MI 48202
4;1 Campus Dr, Allendale, MI 49401
```

As we can see, we have an **ID** for each row and an address on a column **Addr**. 

Let's first read the data into a pandas DataFrame using the `read_csv()` function:

In [1]:
# Import necessary modules
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Filepath
fp = "data/Michigan_colleges.txt"

# Read the data
data = pd.read_csv(fp, sep=";")
data.head()

,id,addr
0,1,"500 S State St, Ann Arbor, MI 48109"
1,2,"220 Trowbridge Rd, East Lansing, MI 48824"
2,3,"42 W Warren Ave, Detroit, MI 48202"
3,4,"1 Campus Dr, Allendale, MI 49401"
4,5,"1903 W Michigan Ave, Kalamazoo, MI 49008"


Now we have our data in a DataFrame and we can geocode our addresses using the the `geopandas.tools.geocode()` function in geopandas that uses geopy library under the hood. The function geocodes a list of addresses (strings) and returns a **GeoDataFrame** with the geocoded result. 

In the following, we import the **geocode()** function and geocode the addresses using Nominatim. Then we pass the addresses to the function from the column **addr**. As discussed earlier, we need to provide a custom string (name of your application) in the **user_agent** parameter to identify ourselves. We also use the **timeout**-parameter to specify how many seconds to wait for a response from the service.

In our example, we will use *Nominatim* as a *geocoding provider*. [*Nominatim*](https://nominatim.org/) is a library and service using OpenStreetMap data, and run by the OpenStreetMap Foundation. 

<div style="border: 1px solid #cce5ff; background-color: #e9f7fd; padding: 15px; border-radius: 5px;">

**Fair-use**

[Nominatim’s terms of use](https://operations.osmfoundation.org/policies/nominatim/) require that users of the service ensure they don’t send more frequent requests than one per second and that a custom **user-agent** string is attached to each query.

Geopandas’ implementation allows us to specify a `user_agent`, and the library also takes care of respecting Nominatim's rate limit.

Looking up an address is a quite expensive database operation. This is why the public and free-to-use Nominatim server sometimes takes slightly longer to respond. In this example, we add a parameter `timeout=10` to wait up to 10 seconds for a response.

</div>

In [2]:
from geopandas.tools import geocode

geocoded = geocode( data["addr"], provider="nominatim", user_agent="GE5515", timeout=10 )
geocoded.head()

,geometry,address
0,POINT (-83.73838 42.28346),"North Ingalls, 500, South State Street, Old Fo..."
1,POINT (-84.48023 42.72085),"Trowbridge Road, River Trail Neighborhood, Eas..."
2,POINT (-83.06515 42.35700),"Welcome Center, 42, West Warren Avenue, Midtow..."
3,POINT (-85.88562 42.96594),"Einstein Bros. Bagels, 1, North Campus Drive, ..."
4,POINT (-85.60983 42.28500),"Metro/Bronco - W. MIchigan Ave, 1903, West Mic..."


As a result we have a **GeoDataFrame** that contains an **address**-column with the geocoded addresses and a **geometry** column containing **Point**-objects representing the geographic locations of the addresses. Notice that these addresses are not the original addresses but those identified by Nominatim. We can join the data from the original text file to the geocoded result to get the address ids and original addresses along. 

In this case, we join the information using the `.join()` function that makes the table join based on index. We do this because the original dataframe and the geocoded output have an identical index and an identical number of rows:

In [ ]:
join = geocoded.join(data)

# join = data.join(geocoded)

join.head()

,geometry,address,id,addr
0,POINT (-83.73838 42.28346),"North Ingalls, 500, South State Street, Old Fo...",1,"500 S State St, Ann Arbor, MI 48109"
1,POINT (-84.48023 42.72085),"Trowbridge Road, River Trail Neighborhood, Eas...",2,"220 Trowbridge Rd, East Lansing, MI 48824"
2,POINT (-83.06515 42.35700),"Welcome Center, 42, West Warren Avenue, Midtow...",3,"42 W Warren Ave, Detroit, MI 48202"
3,POINT (-85.88562 42.96594),"Einstein Bros. Bagels, 1, North Campus Drive, ...",4,"1 Campus Dr, Allendale, MI 49401"
4,POINT (-85.60983 42.28500),"Metro/Bronco - W. MIchigan Ave, 1903, West Mic...",5,"1903 W Michigan Ave, Kalamazoo, MI 49008"


Here we can see the geocoded address (column `address`) and original address (column `addr`) side-by-side and verify that the result looks correct for the first five rows. 

> **Note**
>
> If you perform the join the other way around, i.e., `data.join(geocoded)`, the output would be a `pandas.DataFrame`, not a `geopandas.GeoDataFrame`.

It’s now easy to save the new data set as a geospatial file, e.g., in *GeoPackage* format:

In [4]:
# Output file path
outfp = "data/Michigan_colleges.gpkg"

# Save to GeoPackage
join.to_file(outfp)

That's it. Now we have successfully geocoded those addresses into Points and made a GeoPackage out of them. Nominatim works relatively nicely if you have well defined and well-known addresses such as the ones that we used in this tutorial. In practice, the address needs to exist in the OpenStreetMap database. Sometimes, however, you might want to geocode a "point-of-interest", such as a museum, only based on its name. If the museum name is not on OpenStreetMap, Nominatim won't provide any results for it, but you might be able to geocode the place using some other geocoders.

#### Quiz

Do another round of geocoding with your own list of addresses from anywhere in the world. Are you happy with the result?

In the above example we passed the address column to the geocoding function. [The geopandas documentation of geopandas.tools.geocode](https://geopandas.org/en/stable/docs/reference/api/geopandas.tools.geocode.html#geopandas-tools-geocode) confirms that we should also be able to pass a list of strings to the geocoding tool. So, you should be able to answer this question by defining a list of addresses and using this list as input strings.

In [5]:
# Solution

# Example list of addresses
address_list = [
    "220 Trowbridge Rd, East Lansing, MI 48824, USA",
    "500 S State St, Ann Arbor, MI 48109, USA",
]

# Do the geocoding
geocoded2 = geocode( address_list, provider="nominatim", user_agent="GE5515", timeout=10 )
geocoded2.head()

,geometry,address
0,POINT (-84.48023 42.72085),"Trowbridge Road, River Trail Neighborhood, Eas..."
1,POINT (-83.73838 42.28346),"North Ingalls, 500, South State Street, Old Fo..."


In [6]:
# Check if the result looks correct on a map
# This line of code uses the Geopandas .explore() method, which is a high-level wrapper for Folium. It allows you to transform a static GeoDataFrame into an interactive, scrollable web map directly in your Python environment (like Jupyter).
# max_zoom=12 limits how far you can zoom into the map.
# tiles="CartoDB Positron" defines the basemap. "CartoDB Positron" is a popular choice for data visualization because it is "light" and "minimalist"

geocoded2.explore( color="red", max_zoom=12, marker_kwds=dict(radius=8), tiles="CartoDB Positron")

## Reverse geocoding

Naturally, it is also possible to do geocoding in a reverse manner, meaning that we have a list of points / coordinates and we want to find out the address for these points. In the following, we continue using the same points that we stored previously into the **geocoded** geodataframe, but at this time, we do the reverse geocoding with them. Let's start by selecting only the **geometry** column from the data:


In [7]:
points = geocoded[["geometry"]].copy()
points.head()

,geometry
0,POINT (-83.73838 42.28346)
1,POINT (-84.48023 42.72085)
2,POINT (-83.06515 42.35700)
3,POINT (-85.88562 42.96594)
4,POINT (-85.60983 42.28500)


Now we have a simple GeoDataFrame with only point objects stored into the **geometry** column. To do the reverse geocoding, i.e. finding addresses to these points, we can use the `reverse_geocode()` function of geopandas. 

The `reverse_geocode()` works similarly as the `geocode()` function but accepts as an input either a **GeoSeries**, or a list of **shapely Point** objects. Similarly, you should specify the geocoding provider, the user_agent, and the timeout value. 

In the following, we use the geometries stored in the points geodataframe for reverse geocoding:

In [8]:
from geopandas.tools import reverse_geocode

reverse_geocoded = reverse_geocode(
    points.geometry, provider="nominatim", user_agent="GE5515", timeout=10
)
reverse_geocoded.head()

,geometry,address
0,POINT (-83.73838 42.28346),"North Ingalls, 500, South State Street, Old Fo..."
1,POINT (-84.48023 42.72085),"Trowbridge Road, River Trail Neighborhood, Eas..."
2,POINT (-83.06515 42.35700),"Welcome Center, 42, West Warren Avenue, Midtow..."
3,POINT (-85.88562 42.96594),"Einstein Bros. Bagels, 1, North Campus Drive, ..."
4,POINT (-85.60983 42.28500),"Metro/Bronco - W. MIchigan Ave, 1903, West Mic..."


As a result, we now have the addresses for each point stored into the `address` column which can be useful e.g. if you want to visualize the points on a map and also show the address to these locations as extra information. 